In [1]:
# ============================================================
# CELL 1: Setup — imports, data loading, helpers
# ============================================================

import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, roc_auc_score
from xgboost import XGBClassifier
import time
import os
import warnings
warnings.filterwarnings("ignore")

DATA_PATH = r"..\..\Data\Main\modelling_dataset.csv"
OUTPUT_DIR = r"..\..\Dissertation_Results\Prediction"
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date']).reset_index(drop=True)

# Main features
MAIN_FEATURES = ['sentiment', 'volatility_20d', 'sent_x_vol']

def evaluate(y_true, y_pred, y_prob=None):
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'MCC': matthews_corrcoef(y_true, y_pred),
    }
    if y_prob is not None:
        try:
            metrics['AUC-ROC'] = roc_auc_score(y_true, y_prob)
        except:
            metrics['AUC-ROC'] = np.nan
    return metrics

def run_walk_forward(data, features, all_months, model_type='LR', xgb_params=None):
    """Run walk-forward with expanding window. Returns metrics dict."""
    data = data.copy()
    data['year_month'] = data['date'].dt.to_period('M')
    
    y_true_all, y_pred_all, y_prob_all = [], [], []
    
    for month in all_months:
        train = data[data['year_month'] < month]
        test = data[data['year_month'] == month]
        
        if len(train) < 100 or len(test) < 10:
            continue
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train[features])
        X_test = scaler.transform(test[features])
        y_train = train['target'].values
        y_test = test['target'].values
        
        if model_type == 'LR':
            model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
            model.fit(X_train, y_train)
        elif model_type == 'XGB':
            n_pos = max(y_train.sum(), 1)
            n_neg = max(len(y_train) - n_pos, 1)
            model = XGBClassifier(
                n_estimators=xgb_params.get('n_estimators', 50),
                max_depth=xgb_params.get('max_depth', 5),
                learning_rate=xgb_params.get('learning_rate', 0.05),
                scale_pos_weight=n_neg / n_pos,
                random_state=42, use_label_encoder=False,
                eval_metric='logloss', verbosity=0
            )
            model.fit(X_train, y_train)
        
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)
        y_prob_all.extend(y_prob)
    
    if len(y_true_all) == 0:
        return {'Accuracy': np.nan, 'F1': np.nan, 'MCC': np.nan, 'AUC-ROC': np.nan}
    
    return evaluate(np.array(y_true_all), np.array(y_pred_all), np.array(y_prob_all))

# XGBoost best params from tuning
XGB_PARAMS = {'n_estimators': 50, 'max_depth': 5, 'learning_rate': 0.05}

print(f"Data loaded: {df.shape[0]:,} rows, {df['ticker'].nunique()} tickers")
print(f"Main features: {MAIN_FEATURES}")
print(f"XGBoost params: {XGB_PARAMS}")
print("Helpers defined.\n")

# Store all rejected approach results
all_rejected_results = []

Data loaded: 589,886 rows, 523 tickers
Main features: ['sentiment', 'volatility_20d', 'sent_x_vol']
XGBoost params: {'n_estimators': 50, 'max_depth': 5, 'learning_rate': 0.05}
Helpers defined.



In [2]:



# ============================================================
# CELL 2: Compute technical indicators
# ============================================================

def compute_rsi(close, window=14):
    delta = close.diff()
    gain = delta.where(delta > 0, 0).rolling(window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

df['rsi_14'] = df.groupby('ticker')['Close'].transform(lambda x: compute_rsi(x, 14))

df['ema_12'] = df.groupby('ticker')['Close'].transform(lambda x: x.ewm(span=12).mean())
df['ema_26'] = df.groupby('ticker')['Close'].transform(lambda x: x.ewm(span=26).mean())
df['macd'] = df['ema_12'] - df['ema_26']

df['ma_5'] = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(5).mean())
df['ma_10'] = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(10).mean())
df['ma5_ma10_ratio'] = df['ma_5'] / df['ma_10']

df['bb_mid'] = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(20).mean())
df['bb_std'] = df.groupby('ticker')['Close'].transform(lambda x: x.rolling(20).std())
df['bb_upper'] = df['bb_mid'] + 2 * df['bb_std']
df['bb_lower'] = df['bb_mid'] - 2 * df['bb_std']
df['bb_position'] = (df['Close'] - df['bb_lower']) / (df['bb_upper'] - df['bb_lower'])

df['vol_avg_20'] = df.groupby('ticker')['Volume'].transform(lambda x: x.rolling(20).mean())
df['volume_ratio'] = df['Volume'] / df['vol_avg_20']

df['momentum_5'] = df.groupby('ticker')['Close'].transform(lambda x: x.pct_change(5))

TECH_COLS = ['rsi_14', 'macd', 'ma5_ma10_ratio', 'bb_position', 'volume_ratio', 'momentum_5']

print("Technical indicators computed:")
for col in TECH_COLS:
    print(f"  {col}: {df[col].isna().sum():,} NaN")


Technical indicators computed:
  rsi_14: 6,799 NaN
  macd: 0 NaN
  ma5_ma10_ratio: 4,707 NaN
  bb_position: 9,937 NaN
  volume_ratio: 9,937 NaN
  momentum_5: 2,615 NaN


In [3]:


# ============================================================
# CELL 3: TEST 1 — Technical Indicators
# Tests: tech only, tech+sentiment, sentiment only, main features
# Estimated time: ~10 minutes
# ============================================================

print("=" * 70)
print("TEST 1: TECHNICAL INDICATORS")
print("=" * 70)

all_needed = ['volatility_20d', 'next_day_return', 'sentiment', 'sent_x_vol'] + TECH_COLS
tech_df = df.dropna(subset=all_needed).copy()
tech_df['year_month'] = tech_df['date'].dt.to_period('M')
all_months = sorted(tech_df[tech_df['date'] >= '2022-01-01']['year_month'].unique())

print(f"Clean dataset: {len(tech_df):,} rows")
print(f"Months: {len(all_months)}\n")

tech_feature_sets = {
    'Main (sent+vol+int)':   MAIN_FEATURES,
    'Technical only':        TECH_COLS,
    'Tech + sentiment':      TECH_COLS + ['sentiment', 'volatility_20d', 'sent_x_vol'],
    'Tech only (no sent)':   TECH_COLS + ['volatility_20d'],
}

print(f"{'Feature Set':<25} {'Model':<6} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 70)

for fs_name, fs_cols in tech_feature_sets.items():
    for model_type in ['LR', 'XGB']:
        t0 = time.time()
        m = run_walk_forward(tech_df, fs_cols, all_months, model_type, XGB_PARAMS)
        elapsed = time.time() - t0
        print(f"{fs_name:<25} {model_type:<6} {m['Accuracy']:>8.4f} {m['F1']:>8.4f} {m['MCC']:>8.4f} {m['AUC-ROC']:>8.4f} [{elapsed:.0f}s]")
        
        all_rejected_results.append({
            'test': 'Technical Indicators',
            'feature_set': fs_name,
            'model': model_type,
            **m
        })

baseline = tech_df[tech_df['date'] >= '2022-01-01']['target'].mean()
print(f"\nBaseline: {baseline:.4f}")


TEST 1: TECHNICAL INDICATORS
Clean dataset: 578,903 rows
Months: 48

Feature Set               Model       Acc       F1      MCC      AUC
----------------------------------------------------------------------
Main (sent+vol+int)       LR       0.4969   0.4926  -0.0049   0.4962 [8s]
Main (sent+vol+int)       XGB      0.4963   0.5170  -0.0091   0.4947 [17s]
Technical only            LR       0.5022   0.5301   0.0019   0.5024 [13s]
Technical only            XGB      0.5009   0.5491  -0.0035   0.4976 [18s]
Tech + sentiment          LR       0.4999   0.5181  -0.0016   0.5009 [15s]
Tech + sentiment          XGB      0.4990   0.5320  -0.0053   0.4974 [19s]
Tech only (no sent)       LR       0.5011   0.5195   0.0009   0.5018 [13s]
Tech only (no sent)       XGB      0.5005   0.5428  -0.0034   0.4989 [19s]

Baseline: 0.5141


In [4]:
# ============================================================
# CELL 4: TEST 2 — Rolling Windows (6m, 12m, 18m)
# Instead of expanding, use fixed-size rolling training window
# Estimated time: ~10 minutes
# ============================================================

print("=" * 70)
print("TEST 2: ROLLING WINDOWS")
print("=" * 70)

model_df = df.dropna(subset=['volatility_20d', 'next_day_return']).copy()
model_df = model_df.sort_values(['ticker', 'date']).reset_index(drop=True)
model_df['year_month'] = model_df['date'].dt.to_period('M')
all_months = sorted(model_df[model_df['date'] >= '2022-01-01']['year_month'].unique())

rolling_windows = {
    '6-month rolling':  6,
    '12-month rolling': 12,
    '18-month rolling': 18,
    'Expanding (main)': None,  # None = expanding
}

print(f"{'Window':<25} {'Model':<6} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 70)

for window_name, window_months in rolling_windows.items():
    for model_type in ['LR', 'XGB']:
        t0 = time.time()
        y_true_all, y_pred_all, y_prob_all = [], [], []
        
        for month in all_months:
            if window_months is None:
                # Expanding: all data before this month
                train = model_df[model_df['year_month'] < month]
            else:
                # Rolling: only last N months before this month
                month_ts = month.to_timestamp()
                train_start = month_ts - pd.DateOffset(months=window_months)
                train = model_df[(model_df['date'] >= train_start) & (model_df['year_month'] < month)]
            
            test = model_df[model_df['year_month'] == month]
            
            if len(train) < 100 or len(test) < 10:
                continue
            
            scaler = StandardScaler()
            X_train = scaler.fit_transform(train[MAIN_FEATURES])
            X_test = scaler.transform(test[MAIN_FEATURES])
            y_train = train['target'].values
            y_test = test['target'].values
            
            if model_type == 'LR':
                model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
            else:
                n_pos = max(y_train.sum(), 1)
                n_neg = max(len(y_train) - n_pos, 1)
                model = XGBClassifier(
                    **XGB_PARAMS, scale_pos_weight=n_neg/n_pos,
                    random_state=42, use_label_encoder=False,
                    eval_metric='logloss', verbosity=0
                )
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1]
            
            y_true_all.extend(y_test)
            y_pred_all.extend(y_pred)
            y_prob_all.extend(y_prob)
        
        y_true_all = np.array(y_true_all)
        y_pred_all = np.array(y_pred_all)
        y_prob_all = np.array(y_prob_all)
        
        m = evaluate(y_true_all, y_pred_all, y_prob_all)
        elapsed = time.time() - t0
        print(f"{window_name:<25} {model_type:<6} {m['Accuracy']:>8.4f} {m['F1']:>8.4f} {m['MCC']:>8.4f} {m['AUC-ROC']:>8.4f} [{elapsed:.0f}s]")
        
        all_rejected_results.append({
            'test': 'Rolling Windows',
            'feature_set': window_name,
            'model': model_type,
            **m
        })



TEST 2: ROLLING WINDOWS
Window                    Model       Acc       F1      MCC      AUC
----------------------------------------------------------------------
6-month rolling           LR       0.4969   0.4836  -0.0040   0.4983 [2s]
6-month rolling           XGB      0.5004   0.5061   0.0010   0.5003 [4s]
12-month rolling          LR       0.4933   0.4521  -0.0086   0.4958 [3s]
12-month rolling          XGB      0.4971   0.5063  -0.0061   0.4967 [6s]
18-month rolling          LR       0.4915   0.4838  -0.0153   0.4930 [4s]
18-month rolling          XGB      0.4989   0.5072  -0.0023   0.4977 [7s]
Expanding (main)          LR       0.4969   0.4926  -0.0049   0.4962 [7s]
Expanding (main)          XGB      0.4963   0.5170  -0.0091   0.4947 [15s]


In [5]:



# ============================================================
# CELL 5: TEST 3 — Different Base Training Periods
# Instead of always starting from 2020, try different starts
# Estimated time: ~10 minutes
# ============================================================

print("=" * 70)
print("TEST 3: DIFFERENT BASE TRAINING PERIODS")
print("=" * 70)

base_periods = {
    '2020-2021 (main)':  ('2020-01-01', '2021-12-31', '2022-01-01'),
    '2020 only':         ('2020-01-01', '2020-12-31', '2021-01-01'),
    '2021 only':         ('2021-01-01', '2021-12-31', '2022-01-01'),
    '2020-2022':         ('2020-01-01', '2022-12-31', '2023-01-01'),
    '2020-2023':         ('2020-01-01', '2023-12-31', '2024-01-01'),
}

print(f"{'Base Period':<25} {'Model':<6} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8} {'Test Months':>12}")
print("-" * 80)

for period_name, (train_start, train_end, test_start) in base_periods.items():
    test_months = sorted(model_df[model_df['date'] >= test_start]['year_month'].unique())
    
    if len(test_months) == 0:
        print(f"{period_name:<25} — no test data")
        continue
    
    for model_type in ['LR', 'XGB']:
        t0 = time.time()
        y_true_all, y_pred_all, y_prob_all = [], [], []
        
        for month in test_months:
            # Expanding from the base period start
            train = model_df[(model_df['date'] >= train_start) & (model_df['year_month'] < month)]
            test = model_df[model_df['year_month'] == month]
            
            if len(train) < 100 or len(test) < 10:
                continue
            
            scaler = StandardScaler()
            X_train = scaler.fit_transform(train[MAIN_FEATURES])
            X_test = scaler.transform(test[MAIN_FEATURES])
            y_train = train['target'].values
            y_test = test['target'].values
            
            if model_type == 'LR':
                model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
            else:
                n_pos = max(y_train.sum(), 1)
                n_neg = max(len(y_train) - n_pos, 1)
                model = XGBClassifier(
                    **XGB_PARAMS, scale_pos_weight=n_neg/n_pos,
                    random_state=42, use_label_encoder=False,
                    eval_metric='logloss', verbosity=0
                )
            
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_prob = model.predict_proba(X_test)[:, 1]
            
            y_true_all.extend(y_test)
            y_pred_all.extend(y_pred)
            y_prob_all.extend(y_prob)
        
        y_true_all = np.array(y_true_all)
        y_pred_all = np.array(y_pred_all)
        y_prob_all = np.array(y_prob_all)
        
        m = evaluate(y_true_all, y_pred_all, y_prob_all)
        elapsed = time.time() - t0
        print(f"{period_name:<25} {model_type:<6} {m['Accuracy']:>8.4f} {m['F1']:>8.4f} {m['MCC']:>8.4f} {m['AUC-ROC']:>8.4f} {len(test_months):>12} [{elapsed:.0f}s]")
        
        all_rejected_results.append({
            'test': 'Different Base Periods',
            'feature_set': period_name,
            'model': model_type,
            **m
        })



TEST 3: DIFFERENT BASE TRAINING PERIODS
Base Period               Model       Acc       F1      MCC      AUC  Test Months
--------------------------------------------------------------------------------
2020-2021 (main)          LR       0.4969   0.4926  -0.0049   0.4962           48 [8s]
2020-2021 (main)          XGB      0.4963   0.5170  -0.0091   0.4947           48 [15s]
2020 only                 LR       0.4978   0.5088  -0.0046   0.4955           60 [8s]
2020 only                 XGB      0.4972   0.5282  -0.0092   0.4943           60 [17s]
2021 only                 LR       0.5022   0.5423   0.0002   0.4972           48 [6s]
2021 only                 XGB      0.4986   0.5321  -0.0061   0.4957           48 [13s]
2020-2022                 LR       0.4967   0.4737  -0.0008   0.4974           36 [6s]
2020-2022                 XGB      0.4961   0.5346  -0.0132   0.4916           36 [13s]
2020-2023                 LR       0.4980   0.4853   0.0008   0.4990           24 [5s]
2020-2023 

In [6]:

# ============================================================
# CELL 6: TEST 4 — Sector-Level Models
# Train one model per GICS sector instead of pooled/per-stock
# Estimated time: ~10 minutes
# ============================================================

print("=" * 70)
print("TEST 4: SECTOR-LEVEL MODELS")
print("=" * 70)

# Load sector information from panel or assign via first two digits of GICS
# If GICS not available, use ticker-based sector mapping
# For now, use a simpler approach: download sector info

# Check if we have sector data
if 'sector' not in model_df.columns:
    print("No sector column in data. Attempting to create from ticker groupings...")
    # Use the membership panel to get sector info if available
    try:
        panel = pd.read_csv(r"..\..\Data\Main\membership_panel_clean.csv")
        if 'sector' in panel.columns:
            sector_map = panel.set_index('yf_ticker')['sector'].to_dict()
            model_df['sector'] = model_df['ticker'].map(sector_map)
            print(f"Sectors mapped: {model_df['sector'].nunique()} sectors")
        else:
            print("No sector column in panel either.")
            print("Assigning sectors via yfinance (slow) or skipping...")
            
            # Quick alternative: use first letter grouping as proxy
            # Better: try to load from a pre-saved sector file
            sector_file = r"..\..\Data\Main\ticker_sectors.csv"
            if os.path.exists(sector_file):
                sectors = pd.read_csv(sector_file)
                sector_map = sectors.set_index('ticker')['sector'].to_dict()
                model_df['sector'] = model_df['ticker'].map(sector_map)
                print(f"Sectors loaded from file: {model_df['sector'].nunique()} sectors")
            else:
                print("No sector file found. Downloading from yfinance...")
                import yfinance as yf
                tickers = model_df['ticker'].unique()
                sector_data = []
                for i, t in enumerate(tickers):
                    try:
                        info = yf.Ticker(t).info
                        sector_data.append({'ticker': t, 'sector': info.get('sector', 'Unknown')})
                    except:
                        sector_data.append({'ticker': t, 'sector': 'Unknown'})
                    if (i+1) % 50 == 0:
                        print(f"  Downloaded {i+1}/{len(tickers)}")
                
                sectors_df = pd.DataFrame(sector_data)
                sectors_df.to_csv(sector_file, index=False)
                sector_map = sectors_df.set_index('ticker')['sector'].to_dict()
                model_df['sector'] = model_df['ticker'].map(sector_map)
                print(f"Sectors downloaded and saved: {model_df['sector'].nunique()} sectors")
    except Exception as e:
        print(f"Could not load sectors: {e}")
        model_df['sector'] = 'Unknown'

all_months_sector = sorted(model_df[model_df['date'] >= '2022-01-01']['year_month'].unique())

if 'sector' in model_df.columns and model_df['sector'].nunique() > 1:
    sectors = model_df['sector'].dropna().unique()
    print(f"\nSectors found: {len(sectors)}")
    for s in sorted(sectors):
        count = (model_df['sector'] == s).sum()
        print(f"  {s}: {count:,} rows")
    
    # Run sector-level models
    print(f"\n{'Approach':<25} {'Model':<6} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
    print("-" * 70)
    
    for model_type in ['LR', 'XGB']:
        t0 = time.time()
        y_true_all, y_pred_all, y_prob_all = [], [], []
        
        for month in all_months_sector:
            for sector in sectors:
                sector_data = model_df[model_df['sector'] == sector]
                train = sector_data[sector_data['year_month'] < month]
                test = sector_data[sector_data['year_month'] == month]
                
                if len(train) < 100 or len(test) < 5:
                    continue
                
                scaler = StandardScaler()
                X_train = scaler.fit_transform(train[MAIN_FEATURES])
                X_test = scaler.transform(test[MAIN_FEATURES])
                y_train = train['target'].values
                y_test = test['target'].values
                
                if model_type == 'LR':
                    model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
                else:
                    n_pos = max(y_train.sum(), 1)
                    n_neg = max(len(y_train) - n_pos, 1)
                    model = XGBClassifier(
                        **XGB_PARAMS, scale_pos_weight=n_neg/n_pos,
                        random_state=42, use_label_encoder=False,
                        eval_metric='logloss', verbosity=0
                    )
                
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                y_prob = model.predict_proba(X_test)[:, 1]
                
                y_true_all.extend(y_test)
                y_pred_all.extend(y_pred)
                y_prob_all.extend(y_prob)
        
        y_true_all = np.array(y_true_all)
        y_pred_all = np.array(y_pred_all)
        y_prob_all = np.array(y_prob_all)
        
        m = evaluate(y_true_all, y_pred_all, y_prob_all)
        elapsed = time.time() - t0
        print(f"{'Sector-level':<25} {model_type:<6} {m['Accuracy']:>8.4f} {m['F1']:>8.4f} {m['MCC']:>8.4f} {m['AUC-ROC']:>8.4f} [{elapsed:.0f}s]")
        
        all_rejected_results.append({
            'test': 'Sector-Level Models',
            'feature_set': 'Sector-level',
            'model': model_type,
            **m
        })
    
    # Also run pooled (main approach) on same data for comparison
    for model_type in ['LR', 'XGB']:
        m = run_walk_forward(model_df.dropna(subset=MAIN_FEATURES), MAIN_FEATURES, all_months_sector, model_type, XGB_PARAMS)
        print(f"{'Pooled (main)':<25} {model_type:<6} {m['Accuracy']:>8.4f} {m['F1']:>8.4f} {m['MCC']:>8.4f} {m['AUC-ROC']:>8.4f}")
else:
    print("Sector data not available. Skipping sector-level test.")
    print("If you want this test, create a ticker_sectors.csv with columns: ticker, sector")


TEST 4: SECTOR-LEVEL MODELS
No sector column in data. Attempting to create from ticker groupings...
No sector column in panel either.
Assigning sectors via yfinance (slow) or skipping...
Sectors loaded from file: 12 sectors

Sectors found: 12
  Basic Materials: 25,334 rows
  Communication Services: 27,480 rows
  Consumer Cyclical: 72,154 rows
  Consumer Defensive: 44,626 rows
  Energy: 25,082 rows
  Financial Services: 80,472 rows
  Healthcare: 71,617 rows
  Industrials: 78,805 rows
  Real Estate: 32,696 rows
  Technology: 84,950 rows
  Unknown: 1,206 rows
  Utilities: 34,481 rows

Approach                  Model       Acc       F1      MCC      AUC
----------------------------------------------------------------------
Sector-level              LR       0.4974   0.5089  -0.0057   0.4953 [22s]
Sector-level              XGB      0.4996   0.5101  -0.0012   0.4993 [50s]
Pooled (main)             LR       0.4969   0.4926  -0.0049   0.4962
Pooled (main)             XGB      0.4963   0.5170  

In [7]:

# ============================================================
# CELL 7: TEST 5 — Per-Stock Single Split (No Walk-Forward)
# This tests whether apparent per-stock accuracy is overfitting
# Estimated time: ~5 minutes
# ============================================================

print("=" * 70)
print("TEST 5: PER-STOCK SINGLE SPLIT (NO WALK-FORWARD)")
print("=" * 70)

# Single split: train 2020-2021, test 2022-2025 (one shot, no retraining)
train_single = model_df[model_df['date'] < '2022-01-01']
test_single = model_df[model_df['date'] >= '2022-01-01']

tickers = model_df['ticker'].unique()

print(f"Train: {len(train_single):,} rows")
print(f"Test: {len(test_single):,} rows")
print(f"Tickers: {len(tickers)}\n")

for model_type in ['LR', 'XGB']:
    t0 = time.time()
    y_true_all, y_pred_all, y_prob_all = [], [], []
    per_stock_accs = []
    
    for t in tickers:
        train_t = train_single[train_single['ticker'] == t]
        test_t = test_single[test_single['ticker'] == t]
        
        if len(train_t) < 50 or len(test_t) < 20:
            continue
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(train_t[MAIN_FEATURES])
        X_test = scaler.transform(test_t[MAIN_FEATURES])
        y_train = train_t['target'].values
        y_test = test_t['target'].values
        
        if model_type == 'LR':
            model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
        else:
            n_pos = max(y_train.sum(), 1)
            n_neg = max(len(y_train) - n_pos, 1)
            model = XGBClassifier(
                **XGB_PARAMS, scale_pos_weight=n_neg/n_pos,
                random_state=42, use_label_encoder=False,
                eval_metric='logloss', verbosity=0
            )
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]
        
        stock_acc = accuracy_score(y_test, y_pred)
        per_stock_accs.append(stock_acc)
        
        y_true_all.extend(y_test)
        y_pred_all.extend(y_pred)
        y_prob_all.extend(y_prob)
    
    y_true_all = np.array(y_true_all)
    y_pred_all = np.array(y_pred_all)
    y_prob_all = np.array(y_prob_all)
    per_stock_accs = np.array(per_stock_accs)
    
    m = evaluate(y_true_all, y_pred_all, y_prob_all)
    elapsed = time.time() - t0
    
    print(f"\n{model_type} — Single Split Per-Stock:")
    print(f"  Aggregate:  Acc={m['Accuracy']:.4f}, F1={m['F1']:.4f}, MCC={m['MCC']:.4f}, AUC={m['AUC-ROC']:.4f}")
    print(f"  Per-stock accuracy: mean={per_stock_accs.mean():.4f}, min={per_stock_accs.min():.4f}, max={per_stock_accs.max():.4f}")
    print(f"  Stocks above 55%: {(per_stock_accs > 0.55).sum()} / {len(per_stock_accs)}")
    print(f"  Stocks above 58%: {(per_stock_accs > 0.58).sum()} / {len(per_stock_accs)}")
    print(f"  Stocks above 60%: {(per_stock_accs > 0.60).sum()} / {len(per_stock_accs)}")
    print(f"  [{elapsed:.0f}s]")
    
    all_rejected_results.append({
        'test': 'Per-Stock Single Split',
        'feature_set': 'Single split (no walk-forward)',
        'model': model_type,
        **m,
        'max_per_stock_acc': per_stock_accs.max(),
        'stocks_above_55': (per_stock_accs > 0.55).sum(),
    })



TEST 5: PER-STOCK SINGLE SPLIT (NO WALK-FORWARD)
Train: 165,150 rows
Test: 413,753 rows
Tickers: 523


LR — Single Split Per-Stock:
  Aggregate:  Acc=0.5018, F1=0.5106, MCC=0.0034, AUC=0.5016
  Per-stock accuracy: mean=0.5023, min=0.4345, max=0.5965
  Stocks above 55%: 7 / 464
  Stocks above 58%: 1 / 464
  Stocks above 60%: 0 / 464
  [13s]

XGB — Single Split Per-Stock:
  Aggregate:  Acc=0.5005, F1=0.5133, MCC=0.0004, AUC=0.4999
  Per-stock accuracy: mean=0.4999, min=0.3509, max=0.5572
  Stocks above 55%: 2 / 464
  Stocks above 58%: 0 / 464
  Stocks above 60%: 0 / 464
  [28s]


In [8]:

# ============================================================
# CELL 8: TEST 6 — Enhanced Sentiment Features
# Add lagged sentiment, sentiment change, abs sentiment
# Estimated time: ~5 minutes
# ============================================================

print("=" * 70)
print("TEST 6: ENHANCED SENTIMENT FEATURES")
print("=" * 70)

# Compute enhanced sentiment features
df['sent_lag1'] = df.groupby('ticker')['sentiment'].shift(1)
df['sent_lag2'] = df.groupby('ticker')['sentiment'].shift(2)
df['sent_change'] = df['sentiment'] - df['sent_lag1']
df['abs_sentiment'] = df['sentiment'].abs()
df['sent_rolling3'] = df.groupby('ticker')['sentiment'].transform(lambda x: x.rolling(3).mean())

ENHANCED_SENT = ['sentiment', 'volatility_20d', 'sent_x_vol', 
                 'sent_lag1', 'sent_lag2', 'sent_change', 'abs_sentiment', 'sent_rolling3']

enhanced_df = df.dropna(subset=ENHANCED_SENT + ['next_day_return']).copy()
enhanced_df['year_month'] = enhanced_df['date'].dt.to_period('M')
all_months_enh = sorted(enhanced_df[enhanced_df['date'] >= '2022-01-01']['year_month'].unique())

print(f"Enhanced features: {ENHANCED_SENT}")
print(f"Clean dataset: {len(enhanced_df):,} rows\n")

enh_feature_sets = {
    'Main features':       MAIN_FEATURES,
    'Enhanced sentiment':  ENHANCED_SENT,
}

print(f"{'Feature Set':<25} {'Model':<6} {'Acc':>8} {'F1':>8} {'MCC':>8} {'AUC':>8}")
print("-" * 70)

for fs_name, fs_cols in enh_feature_sets.items():
    for model_type in ['LR', 'XGB']:
        t0 = time.time()
        m = run_walk_forward(enhanced_df, fs_cols, all_months_enh, model_type, XGB_PARAMS)
        elapsed = time.time() - t0
        print(f"{fs_name:<25} {model_type:<6} {m['Accuracy']:>8.4f} {m['F1']:>8.4f} {m['MCC']:>8.4f} {m['AUC-ROC']:>8.4f} [{elapsed:.0f}s]")
        
        all_rejected_results.append({
            'test': 'Enhanced Sentiment',
            'feature_set': fs_name,
            'model': model_type,
            **m
        })



TEST 6: ENHANCED SENTIMENT FEATURES
Enhanced features: ['sentiment', 'volatility_20d', 'sent_x_vol', 'sent_lag1', 'sent_lag2', 'sent_change', 'abs_sentiment', 'sent_rolling3']
Clean dataset: 578,903 rows

Feature Set               Model       Acc       F1      MCC      AUC
----------------------------------------------------------------------
Main features             LR       0.4969   0.4926  -0.0049   0.4962 [8s]
Main features             XGB      0.4963   0.5170  -0.0091   0.4947 [16s]
Enhanced sentiment        LR       0.5041   0.5733  -0.0003   0.4977 [11s]
Enhanced sentiment        XGB      0.4991   0.5241  -0.0040   0.4968 [20s]


In [9]:


# ============================================================
# CELL 9: Summary of ALL Rejected Approaches
# ============================================================

print("=" * 90)
print("SUMMARY: ALL REJECTED APPROACHES")
print("=" * 90)

rejected_df = pd.DataFrame(all_rejected_results)

print(f"\n{'Test':<30} {'Feature Set':<30} {'Model':<6} {'Acc':>8} {'MCC':>8}")
print("-" * 90)
for _, row in rejected_df.iterrows():
    print(f"{row['test']:<30} {row['feature_set']:<30} {row['model']:<6} {row['Accuracy']:>8.4f} {row['MCC']:>8.4f}")

rejected_df.to_csv(os.path.join(OUTPUT_DIR, "rejected_approaches_results.csv"), index=False)
print(f"\nSaved: rejected_approaches_results.csv")

print(f"\n=== VERDICT ===")
best_acc = rejected_df['Accuracy'].max()
best_mcc = rejected_df['MCC'].max()
print(f"Best accuracy across all rejected approaches: {best_acc:.4f}")
print(f"Best MCC across all rejected approaches: {best_mcc:.4f}")
print(f"Baseline: {model_df[model_df['date'] >= '2022-01-01']['target'].mean():.4f}")

if best_acc < 0.52:
    print("\nNone of the rejected approaches meaningfully exceed random prediction.")
else:
    print(f"\nSome approaches exceeded 52% — check which ones and whether they hold under walk-forward.")

SUMMARY: ALL REJECTED APPROACHES

Test                           Feature Set                    Model       Acc      MCC
------------------------------------------------------------------------------------------
Technical Indicators           Main (sent+vol+int)            LR       0.4969  -0.0049
Technical Indicators           Main (sent+vol+int)            XGB      0.4963  -0.0091
Technical Indicators           Technical only                 LR       0.5022   0.0019
Technical Indicators           Technical only                 XGB      0.5009  -0.0035
Technical Indicators           Tech + sentiment               LR       0.4999  -0.0016
Technical Indicators           Tech + sentiment               XGB      0.4990  -0.0053
Technical Indicators           Tech only (no sent)            LR       0.5011   0.0009
Technical Indicators           Tech only (no sent)            XGB      0.5005  -0.0034
Rolling Windows                6-month rolling                LR       0.4969  -0.0040
Rolli